# Stabilizer Approach Experiment

This experiment aims to explore the difference in fidelity between different approaches for quantum error correction.

## Tasks:

- [ ] Test with Mirage decomposition of syndromes
- [ ] Add more physical assumptions to how long gates / measurements take to perform
- Test with different Topologies
    - [ ] Add functionality to experiment_run to allow remeasuring of qubits
    - [ ] Add functionality to circuits to allow remeasuring old ancilla to use in syndrome measurement
    - [ ] Determine reset time for ancilla to be reused after measurement
    
- Determine whether can use Dylan's gradient flow based evolution to determine any physical limits which might come up as future problems
- Determine how Mirage can be implemented properly with what code


In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
# Import the compact ExperimentRunner and required libs
import os
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from quantum_logical.circuits import (
    parallel_erasure_circuit,
    parallel_erasure_SqrtISWAP_circuit,
    partial_erasure_circuit,
    serial_erasure_circuit,
    serial_erasure_SqrtISWAP_circuit,
)
from quantum_logical.experiment_runner import ExperimentRunner
from quantum_logical.gates import (
    cnot_sqrt_iswap_decomposition,
    hadamard_operator,
    sqrt_iswap_qutrit,
)
from quantum_logical.phase_circuits import (
    parallel_phase_circuit,
    parallel_phase_SqrtISWAP_circuit,
    partial_phase_circuit,
    serial_phase_circuit,
    serial_phase_SqrtISWAP_circuit,
)

runner = ExperimentRunner()


In [ ]:
# Test sqrtISWAP on all basis states
a00 = qt.tensor(qt.basis(3, 0), qt.basis(3, 0)) #|00>
a01 = qt.tensor(qt.basis(3, 0), qt.basis(3, 1)) #|01>
a02 = qt.tensor(qt.basis(3, 0), qt.basis(3, 2)) #|02>
a10 = qt.tensor(qt.basis(3, 1), qt.basis(3, 0)) #|10>
a11 = qt.tensor(qt.basis(3, 1), qt.basis(3, 1)) #|11>
a12 = qt.tensor(qt.basis(3, 1), qt.basis(3, 1)) #|12>
a20 = qt.tensor(qt.basis(3, 2), qt.basis(3, 0)) #|20>
a21 = qt.tensor(qt.basis(3, 2), qt.basis(3, 1)) #|21>
a22 = qt.tensor(qt.basis(3, 2), qt.basis(3, 2)) #|22>


sqrt_iswap = sqrt_iswap_qutrit(2, 0, 1)

print(sqrt_iswap * a00) # |00>
print(sqrt_iswap * a01) # (1/sqrt(2))|01> + i(1/sqrt(2))|10>
print(sqrt_iswap * a02) # |02>
print(sqrt_iswap * a10) # (1/sqrt(2))|10> + i(1/sqrt(2))|01>
print(sqrt_iswap * a11) # |11>
print(sqrt_iswap * a12) # |12>
print(sqrt_iswap * a20) # |20>
print(sqrt_iswap * a21) # |21>
print(sqrt_iswap * a22) # |22>

In [ ]:
cnot_circuit = cnot_sqrt_iswap_decomposition(0,1,2,3)

b00 = a00
b01 = a01
b02 = a02
b10 = a10
b11 = a11
b12 = a12
b20 = a20
b21 = a21
b22 = a22

for gate in cnot_circuit:
    b00 = gate * b00
    b01 = gate * b01
    b02 = gate * b02
    b10 = gate * b10
    b11 = gate * b11
    b12 = gate * b12
    b20 = gate * b20
    b21 = gate * b21
    b22 = gate * b22

print(f"Should be {base_three_to_base_ten([0,0])}", b00) # |00>
print(b01) # |01>
print(b02) # |02>
print(b10) # |11>
print(b11) # |10>
print(b12) # |12>
print(b20) # |20>
print(b21) # |21>
print(b22) # |22>

In [ ]:
cnot_circuit = cnot_sqrt_iswap_decomposition(0,1,2,3)

b00 = a00
b01 = a01
b02 = a02
b10 = a10
b11 = a11
b12 = a12
b20 = a20
b21 = a21
b22 = a22

for gate in cnot_circuit:
    b00 = gate * b00
    b01 = gate * b01
    b02 = gate * b02
    b10 = gate * b10
    b11 = gate * b11
    b12 = gate * b12
    b20 = gate * b20
    b21 = gate * b21
    b22 = gate * b22

print(f"Should be {base_three_to_base_ten([0,0])}", b00) # |00>
print(b01) # |01>
print(b02) # |02>
print(b10) # |11>
print(b11) # |10>
print(b12) # |12>
print(b20) # |20>
print(b21) # |21>
print(b22) # |22>

In [ ]:
cnot_circuit = cnot_sqrt_iswap_decomposition(0,1,2,3)

b00 = a00
b01 = a01
b02 = a02
b10 = a10
b11 = a11
b12 = a12
b20 = a20
b21 = a21
b22 = a22

for gate in cnot_circuit:
    b00 = gate * b00
    b01 = gate * b01
    b02 = gate * b02
    b10 = gate * b10
    b11 = gate * b11
    b12 = gate * b12
    b20 = gate * b20
    b21 = gate * b21
    b22 = gate * b22

print(f"00 Should be {base_three_to_base_ten([0,0])}", b00) # |00>
print(f"01 Should be {base_three_to_base_ten([0,1])}", b01) # |01>
print(f"02 Should be {base_three_to_base_ten([0,2])}", b02) # |02>
print(f"10 Should be {base_three_to_base_ten([1,1])}", b10) # |11>
print(f"11 Should be {base_three_to_base_ten([1,0])}", b11) # |10>
print(f"12 Should be {base_three_to_base_ten([1,2])}", b12) # |12>
print(f"20 Should be {base_three_to_base_ten([2,0])}", b20) # |20>
print(f"21 Should be {base_three_to_base_ten([2,1])}", b21) # |21>
print(f"22 Should be {base_three_to_base_ten([2,2])}", b22) # |22>

In [7]:
def base_three_to_base_ten(digits: List[int]) -> int:
    """Convert a list of base-3 digits to a base-10 integer."""
    return sum(d * (3 ** i) for i, d in enumerate(reversed(digits)))

In [8]:
def plot_results(trotter_dt: float, state_list: List[qt.Qobj], output_path: str) -> None:
    """Create and save diagnostic plots for a completed experiment.

    The function computes time-steps from `trotter_dt` and extracts marginal
    probabilities and fidelities from `state_list`. It writes three PNG files
    (`probabilities_plot.png`, `fidelities_plot.png`, `ancillary_probabilities_plot.png`) into
    `output_path` and also displays them via `matplotlib`.

    Args:
        trotter_dt: Time resolution used during simulation (used to label x-axis).
        state_list: Ordered list of `qutip.Qobj` states from the simulation.
        output_path: Directory where PNGs will be written (created if missing).

    Returns:
        None
    """
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    time_steps = np.linspace(0, trotter_dt * (len(state_list)), len(state_list))

    probabilities = [state.ptrace([0, 1, 2]).diag().real for state in state_list]
    probabilities = np.array(probabilities)

    #init_state = (
    #    qt.tensor(qt.basis(3, 0), qt.basis(3, 0), qt.basis(3, 0))
    #    + qt.tensor(qt.basis(3, 0), qt.basis(3, 2), qt.basis(3, 2))
    #    + qt.tensor(qt.basis(3, 2), qt.basis(3, 2), qt.basis(3, 0))
    #    + qt.tensor(qt.basis(3, 2), qt.basis(3, 0), qt.basis(3, 2))
    #) / 2.0

    plus_state = hadamard_operator(3) * qt.basis(3,0)
    init_state = qt.tensor(plus_state, plus_state, plus_state)
    fidelities = [qt.fidelity(state.ptrace([0, 1, 2]), init_state) for state in state_list]
    fidelities = np.array(fidelities)

    plt.figure(figsize=(10, 6))
    for i in range(27): # one 0 two 2, 222
        plt.plot(time_steps, probabilities[:, i], label=f"State |{np.base_repr(i, base=3)}>")
    plt.xlabel("Time Step")
    plt.ylabel("Probability")
    plt.title("State Probabilities Over Time")
    plt.legend()
    plt.savefig(f"{output_path}/probabilities_plot.png")
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.plot(time_steps, fidelities, label="Fidelity")
    plt.xlabel("Time Step")
    plt.ylabel("Fidelities")
    plt.title("Fidelity between |+++> Over Time")
    plt.text(0.5, 0.5, f"Final Fidelity: {fidelities[-1]:.4f}")
    plt.legend()
    plt.savefig(f"{output_path}/fidelities_plot.png")
    plt.show()

    np.save(f"{output_path}/fidelity_array.npy", fidelities)

    probabilities = [state.ptrace([3]).diag().real for state in state_list]
    probabilities = np.array(probabilities)
    plt.figure(figsize=(10, 6))
    for i in [0, 1, 2]: # one 0 two 2, 222
        plt.plot(time_steps, probabilities[:, i], label=f"State {np.base_repr(i, base=3)}")
    plt.xlabel("Time Step")
    plt.ylabel("Probabilities")
    plt.title("Ancillary Probabilities Over Time")
    plt.legend()
    plt.savefig(f"{output_path}/ancillary_probabilities_plot.png")
    plt.show()

In [9]:
# Parameters (kept concise in the notebook)
iterations = 1
t1_list = np.linspace(25, 160, iterations)
t2_list = np.linspace(33.3, 321, iterations)
trotter_dt = .03
cnot_time = .5
single_qudit_time = .03

# Individual superpositions
plus_state = hadamard_operator(3) * qt.basis(3,0)
minus_state = hadamard_operator(3) * qt.basis(3,2)

hada_layer = qt.tensor(hadamard_operator(3), hadamard_operator(3), hadamard_operator(3))
# Prepare initial logical state
logical_psi_0 = (qt.tensor(plus_state, plus_state, plus_state))  # Small  phase error from |+++>
total_psi_0 = qt.tensor(logical_psi_0, qt.basis(3,0))  # ancillas in |0>
rho_0 = total_psi_0 * total_psi_0.dag()

# Create runner and trotterizer, then run experiment with a list of setups
for i in range(iterations):
    trotterizer = runner.generate_trotterizer(trotter_dt, t1_list[i], t2_list[i], 3, 4)
    setups = [partial_erasure_circuit(single_qudit_time, cnot_time, num_ancillae=1, target_state=0, target_ancilla=0), 
              partial_phase_circuit(single_qudit_time, cnot_time, target_state=0),
              partial_erasure_circuit(single_qudit_time, cnot_time, num_ancillae=1, target_state=1, target_ancilla=0),
              partial_phase_circuit(single_qudit_time, cnot_time, target_state=1),
              partial_erasure_circuit(single_qudit_time, cnot_time, num_ancillae=1, target_state=2, target_ancilla=0)]
    state_list = runner.experiment_partial_run(trotterizer, rho_0, setups)
    plot_results(trotter_dt, state_list, "output/test_partial_interleaved")

TypeError: 'int' object is not iterable

: 